# V1-03: LLM Router 개발 기준선

이 notebook은 자연어 기업 리서치 질문을 V1에서 사용할 작업 종류와 인자로 구조화하는 LLM Router의 첫 기준선을 확인합니다. 로컬 `qwen3.6:27b`와 `temperature=0`을 고정합니다.

측정 흐름은 다음과 같습니다.

`질문 → LLM Router → 구조화된 action·인자 → 정답 route 비교 → 정확도 집계`

아직 OpenDART API 호출, LangGraph workflow 실행, Tool 실행, 최종 답변 생성은 하지 않습니다. Router 자체의 판단을 먼저 분리해 봅니다.

## 0. 이 기준선의 범위와 한계

- 15개 질문은 Prompt를 개선하며 확인할 **개발용 사례**입니다. 고정 평가셋·holdout·최종 합격선은 아직 만들지 않았습니다.
- `temperature=0`은 출력 변동을 줄여 Prompt와 schema에 따른 차이를 비교하기 위한 고정 조건입니다. 완전한 결정성을 보장하지는 않습니다.
- 결과가 틀렸다고 해서 곧바로 Graph나 OpenDART Tool이 문제라는 뜻은 아닙니다. 이 notebook은 LLM Router의 action·인자 판단만 측정합니다.
- V1 제외 범위(복수 기업 비교, 주가·거래량, 투자 추천)는 `unsupported`로 명시적으로 구분합니다.
- 구조화 출력 파싱 실패는 실행을 중단하지 않고 오류 사례로 기록합니다.

## 1. 필요한 라이브러리 불러오기

`ChatOllama`는 로컬 Ollama 모델에 연결하고, Pydantic schema는 Router가 반환해야 할 구조를 제한합니다.

In [1]:
from typing import Literal

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

/home/sms/anaconda3/envs/skn25/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Router 출력 계약 정의

`action`은 이후 workflow가 선택할 작업 종류입니다. 나머지 다섯 필드는 후속 OpenDART 조회·계산·공시 검색에 전달할 인자 후보입니다. `recent_year_count`는 “최근 N개 사업연도”를 요청한 경우에만 사용하며, 실제 연도 범위는 실행 기준일의 OpenDART 데이터로 다음 단계에서 결정합니다. 이 notebook에서는 아직 실행하지 않고 값만 비교합니다.

In [2]:
class RouteRequest(BaseModel):
    action: Literal[
        "financial_summary",
        "financial_trend",
        "disclosure_list",
        "financial_and_disclosure",
        "unsupported",
    ]
    company_name: str | None = None
    business_years: list[int] = Field(default_factory=list)
    recent_year_count: int | None = None
    start_date: str | None = None
    end_date: str | None = None

## 3. 로컬 모델과 구조화 출력 준비

`with_structured_output(RouteRequest)`은 일반 문장 대신 `RouteRequest` 형태의 결과를 요청합니다. `temperature=0`은 이번 기준선의 고정 실행 조건입니다. 실행 시각·모델·Prompt 버전도 함께 기록합니다.

In [3]:
from datetime import datetime, timezone
from importlib.metadata import version

MODEL_NAME = "qwen3.6:27b"
TEMPERATURE = 0
PROMPT_VERSION = "v1"

llm = ChatOllama(
    model=MODEL_NAME,
    base_url="http://localhost:11434",
    temperature=TEMPERATURE,
)
structured_router = llm.with_structured_output(RouteRequest)

RUN_CONFIG = {
    "run_at_utc": datetime.now(timezone.utc).isoformat(),
    "model": MODEL_NAME,
    "temperature": TEMPERATURE,
    "prompt_version": PROMPT_VERSION,
    "langchain_version": version("langchain"),
    "langchain_ollama_version": version("langchain-ollama"),
}

RUN_CONFIG

{'run_at_utc': '2026-09-02T09:33:41.828229+00:00',
 'model': 'qwen3.6:27b',
 'temperature': 0,
 'prompt_version': 'v1',
 'langchain_version': '1.2.10',
 'langchain_ollama_version': '1.1.0'}

## 4. Router Prompt

이 Prompt는 자연어 질문을 다섯 action 중 하나로 분류하고, 명시된 기업·연도·날짜만 추출하도록 지시합니다. 나중에는 이 Prompt를 후보별로 바꿔 동일 사례에서 비교할 수 있습니다.

In [4]:
ROUTER_PROMPT = """
당신은 FinSight V1의 질문 Router입니다. 질문을 하나의 action과 인자로 구조화합니다.

action 규칙:
- financial_summary: 한 기업의 특정 사업연도 재무 수치만 요청
- financial_trend: 둘 이상의 명시 사업연도, 성장률, 영업이익률 또는 추세 요청
- disclosure_list: 기간 내 공시 목록 또는 공시 원문 링크만 요청
- financial_and_disclosure: 재무·계산 요청과 공시 근거 요청을 함께 함
- unsupported: 복수 기업 비교, 주가·거래량, 투자 추천 또는 필요한 기업·기간이 없는 요청

인자 규칙:
- company_name에는 질문에 명시된 한 기업명만 넣습니다.
- business_years에는 질문에 명시된 4자리 사업연도만 오름차순으로 넣습니다.
- 최근 N개 사업연도 요청이면 business_years는 빈 목록으로 두고 recent_year_count에 N을 넣습니다.
- start_date와 end_date에는 질문에 명시된 YYYYMMDD 날짜만 넣습니다.
- 추측으로 기업명·연도·날짜를 채우지 않습니다.
- unsupported이면 action 이외 인자는 각각 null, 빈 목록, null, null, null로 둡니다.
"""

## 5. 개발용 정답 사례

각 사례에는 사람이 정한 기대 `action`과 인자가 있습니다. action별 사례를 나누어 적어, 어떤 질문이 어떤 route를 기대하는지 코드에서 직접 읽을 수 있게 합니다.

In [5]:
financial_summary_cases = [
    {"id": "FS-01", "question": "삼성전자의 2024년 매출액, 영업이익, 당기순이익을 알려줘.", "expected": {"action": "financial_summary", "company_name": "삼성전자", "business_years": [2024], "recent_year_count": None, "start_date": None, "end_date": None}},
    {"id": "FS-02", "question": "SK하이닉스 2025년 재무 수치를 조회해줘.", "expected": {"action": "financial_summary", "company_name": "SK하이닉스", "business_years": [2025], "recent_year_count": None, "start_date": None, "end_date": None}},
    {"id": "FS-03", "question": "현대자동차의 2023년 매출과 영업이익은 얼마야?", "expected": {"action": "financial_summary", "company_name": "현대자동차", "business_years": [2023], "recent_year_count": None, "start_date": None, "end_date": None}},
]

In [6]:
financial_trend_cases = [
    {"id": "FT-01", "question": "삼성전자의 최근 3개 사업연도 매출 추세를 정리해줘.", "expected": {"action": "financial_trend", "company_name": "삼성전자", "business_years": [], "recent_year_count": 3, "start_date": None, "end_date": None}},
    {"id": "FT-02", "question": "SK하이닉스의 2023년과 2024년 매출 성장률, 영업이익률을 계산해줘.", "expected": {"action": "financial_trend", "company_name": "SK하이닉스", "business_years": [2023, 2024], "recent_year_count": None, "start_date": None, "end_date": None}},
    {"id": "FT-03", "question": "현대자동차 2023년, 2024년 영업이익률 변화를 알려줘.", "expected": {"action": "financial_trend", "company_name": "현대자동차", "business_years": [2023, 2024], "recent_year_count": None, "start_date": None, "end_date": None}},
]

In [7]:
disclosure_cases = [
    {"id": "DL-01", "question": "삼성전자의 20250101부터 20251231까지 공시 목록과 원문 링크를 알려줘.", "expected": {"action": "disclosure_list", "company_name": "삼성전자", "business_years": [], "recent_year_count": None, "start_date": "20250101", "end_date": "20251231"}},
    {"id": "DL-02", "question": "SK하이닉스 20240101~20241231 공시를 찾아줘.", "expected": {"action": "disclosure_list", "company_name": "SK하이닉스", "business_years": [], "recent_year_count": None, "start_date": "20240101", "end_date": "20241231"}},
    {"id": "DL-03", "question": "현대자동차의 20250101부터 20250630까지 제출된 공시 목록을 보여줘.", "expected": {"action": "disclosure_list", "company_name": "현대자동차", "business_years": [], "recent_year_count": None, "start_date": "20250101", "end_date": "20250630"}},
]

In [8]:
combined_cases = [
    {"id": "FD-01", "question": "삼성전자의 2023년과 2024년 매출, 영업이익률을 정리하고 해당 기간 참고 공시도 알려줘.", "expected": {"action": "financial_and_disclosure", "company_name": "삼성전자", "business_years": [2023, 2024], "recent_year_count": None, "start_date": None, "end_date": None}},
    {"id": "FD-02", "question": "SK하이닉스의 2025년 영업이익과 사업보고서 원문 링크를 알려줘.", "expected": {"action": "financial_and_disclosure", "company_name": "SK하이닉스", "business_years": [2025], "recent_year_count": None, "start_date": None, "end_date": None}},
    {"id": "FD-03", "question": "현대자동차의 2023년과 2024년 매출 변화와 20240101부터 20241231까지 공시를 함께 보여줘.", "expected": {"action": "financial_and_disclosure", "company_name": "현대자동차", "business_years": [2023, 2024], "recent_year_count": None, "start_date": "20240101", "end_date": "20241231"}},
]

In [9]:
unsupported_cases = [
    {"id": "US-01", "question": "삼성전자와 SK하이닉스의 2024년 실적을 비교해줘.", "expected": {"action": "unsupported", "company_name": None, "business_years": [], "recent_year_count": None, "start_date": None, "end_date": None}},
    {"id": "US-02", "question": "삼성전자의 2025년 주가와 거래량을 분석해줘.", "expected": {"action": "unsupported", "company_name": None, "business_years": [], "recent_year_count": None, "start_date": None, "end_date": None}},
    {"id": "US-03", "question": "지금 사도 되는 유망한 국내 주식을 추천해줘.", "expected": {"action": "unsupported", "company_name": None, "business_years": [], "recent_year_count": None, "start_date": None, "end_date": None}},
]

In [10]:
router_cases = (
    financial_summary_cases
    + financial_trend_cases
    + disclosure_cases
    + combined_cases
    + unsupported_cases
)

assert len(router_cases) == 15
print(f"개발용 질문 수: {len(router_cases)}개")

개발용 질문 수: 15개


## 6. Router 호출 함수와 한 질문의 구조화 결과 확인

먼저 호출 실패를 결과로 남기는 작은 함수를 만들고, 한 정상 질문을 넣어 LLM이 일반 문장이 아니라 `action`, 기업명, 사업연도 등의 구조화 값을 반환하는지 확인합니다.

In [11]:
def route_question(question: str) -> dict:
    try:
        response = structured_router.invoke([
            SystemMessage(content=ROUTER_PROMPT),
            HumanMessage(content=question),
        ])
        return {"actual": response.model_dump(), "error": None}
    except Exception as error:
        return {"actual": None, "error": f"{type(error).__name__}: {error}"}


example_case = router_cases[0]
example_run = route_question(example_case["question"])

print("질문:", example_case["question"])
example_run

질문: 삼성전자의 2024년 매출액, 영업이익, 당기순이익을 알려줘.


{'actual': {'action': 'financial_summary',
  'company_name': '삼성전자',
  'business_years': [2024],
  'recent_year_count': None,
  'start_date': None,
  'end_date': None},
 'error': None}

## 7. 모든 개발용 질문 실행

각 질문에서 파싱된 구조화 결과 또는 오류를 저장합니다. action 오류, 인자 오류, 구조화 출력 실패를 나중에 분리해 확인할 수 있습니다.

In [12]:
router_results = []

for case in router_cases:
    router_results.append({**case, **route_question(case["question"])})

print(f"Router 실행 완료: {len(router_results)}개")

Router 실행 완료: 15개


In [13]:
[
    {
        "id": row["id"],
        "expected_action": row["expected"]["action"],
        "actual_action": row["actual"]["action"] if row["actual"] else None,
        "actual_arguments": {key: row["actual"][key] for key in ["company_name", "business_years", "recent_year_count", "start_date", "end_date"]} if row["actual"] else None,
        "error": row["error"],
    }
    for row in router_results
]

[{'id': 'FS-01',
  'expected_action': 'financial_summary',
  'actual_action': 'financial_summary',
  'actual_arguments': {'company_name': '삼성전자',
   'business_years': [2024],
   'recent_year_count': None,
   'start_date': None,
   'end_date': None},
  'error': None},
 {'id': 'FS-02',
  'expected_action': 'financial_summary',
  'actual_action': 'financial_summary',
  'actual_arguments': {'company_name': 'SK하이닉스',
   'business_years': [2025],
   'recent_year_count': None,
   'start_date': None,
   'end_date': None},
  'error': None},
 {'id': 'FS-03',
  'expected_action': 'financial_summary',
  'actual_action': 'financial_summary',
  'actual_arguments': {'company_name': '현대자동차',
   'business_years': [2023],
   'recent_year_count': None,
   'start_date': None,
   'end_date': None},
  'error': None},
 {'id': 'FT-01',
  'expected_action': 'financial_trend',
  'actual_action': 'financial_trend',
  'actual_arguments': {'company_name': '삼성전자',
   'business_years': [],
   'recent_year_count': 3,

## 8. Action/Route Selection Accuracy

질문별로 기대 action과 실제 action이 완전히 같은 비율입니다. 이 지표는 Router가 이후 workflow의 올바른 큰 경로를 선택하는지 봅니다.

In [14]:
action_checks = [
    row["actual"] is not None and row["actual"]["action"] == row["expected"]["action"]
    for row in router_results
]
action_accuracy = sum(action_checks) / len(action_checks)

print(f"Action/Route Selection Accuracy: {action_accuracy:.1%}")

Action/Route Selection Accuracy: 100.0%


## 9. Argument Accuracy

각 질문에서 `company_name`, `business_years`, `recent_year_count`, `start_date`, `end_date` 다섯 인자가 모두 기대값과 같은 비율입니다. action이 맞더라도 연도나 날짜가 틀리면 이 지표에서는 실패입니다.

In [15]:
ARGUMENT_KEYS = ["company_name", "business_years", "recent_year_count", "start_date", "end_date"]
argument_checks = [
    row["actual"] is not None and all(row["actual"][key] == row["expected"][key] for key in ARGUMENT_KEYS)
    for row in router_results
]
argument_accuracy = sum(argument_checks) / len(argument_checks)

print(f"Argument Accuracy: {argument_accuracy:.1%}")

Argument Accuracy: 100.0%


## 10. Unsupported Request Recall

V1 범위 밖 질문만 따로 보아 `unsupported`로 안전하게 분류했는지 확인합니다. 이 값은 전체 정확도가 아니라 지원하지 않는 질문을 놓치지 않은 비율(Recall)입니다. 지원 질문을 잘못 `unsupported`로 버린 사례는 위 Action Accuracy와 실패 목록에서 함께 확인합니다.

In [16]:
unsupported_results = [
    row for row in router_results
    if row["expected"]["action"] == "unsupported"
]
unsupported_checks = [row["actual"] is not None and row["actual"]["action"] == "unsupported" for row in unsupported_results]
unsupported_accuracy = sum(unsupported_checks) / len(unsupported_checks)

print(f"Unsupported Request Recall: {unsupported_accuracy:.1%}")
print(f"지원하지 않는 질문 수: {len(unsupported_results)}개")

Unsupported Request Recall: 100.0%
지원하지 않는 질문 수: 3개


## 11. 구조화 출력 Error Rate와 실패 사례 확인

구조화 출력 실패 비율과 action·인자 실패 사례를 함께 봅니다. 정확도 숫자만으로는 어떤 규칙이 부족한지 알기 어렵기 때문입니다.

In [17]:
error_count = sum(row["error"] is not None for row in router_results)
error_rate = error_count / len(router_results)
print(f"Structured Output Error Rate: {error_rate:.1%} ({error_count}/{len(router_results)})")

failures = [
    {
        "id": row["id"],
        "question": row["question"],
        "expected": row["expected"],
        "actual": row["actual"],
    }
    for row, action_ok, arguments_ok in zip(router_results, action_checks, argument_checks)
    if row["error"] is not None or not action_ok or not arguments_ok
]

print(f"확인할 실패 사례 수: {len(failures)}")
failures

Structured Output Error Rate: 0.0% (0/15)
확인할 실패 사례 수: 0


[]

## 12. 이번 notebook에서 이해할 핵심

1. 구조화 출력은 LLM이 임의 문장을 쓰지 않게 할 뿐, action과 인자의 의미적 정확성을 보장하지는 않습니다. 그래서 정답 route와 비교합니다.
2. Action Accuracy와 Argument Accuracy를 분리하면, 잘못된 원인이 큰 작업 선택인지 세부 인자 추출인지 구분할 수 있습니다.
3. 구조화 출력 실패도 Router 품질의 일부입니다. 오류를 별도 기록해야 정확도와 실패 원인을 함께 해석할 수 있습니다.
4. `temperature=0`은 이번 모델·Prompt 기준선을 비교하기 위한 통제 조건입니다. 운영에서도 Router·Tool 인자 추출처럼 재현성이 중요한 역할에는 유지할 수 있습니다.
5. 이 결과만으로 최종 Prompt·모델·Graph를 채택하지 않습니다. 다음 비교 단계에서 사용자와 고정 평가셋, 합격선, 반복 실행 조건을 정해야 합니다.